# PPE Detection — Continue Training (+20 epochs, gentle refinement)

Continues from your existing **30-epoch `best.pt`** for 20 more epochs at a **low learning rate**, to squeeze a bit more box-localization accuracy (mAP@50-95). mAP@50 is already ~98% and won't move much — this is polish.

### Setup (Settings panel)
1. **Accelerator → GPU T4 x2** · **Internet → On** · *(optional)* Secret `ROBOFLOW_API_KEY`.

### Upload your trained model first
Kaggle needs `best.pt` as an attached input:
1. **kaggle.com → Datasets → New Dataset** → upload your `best.pt` → name it e.g. **`ppe-best`** → Create.
2. In this notebook: **Add Input** (right panel) → search **`ppe-best`** → Add. It mounts under `/kaggle/input/`.

### Run
Run cells 1–6 to confirm the data + model are found, then **Save Version → Save & Run All (Commit)** to refine offline (~2.5 h). Grab the new `best.pt` from the **Output** tab when done.

> Simpler alternative (no upload): in the original `kaggle_ppe.ipynb`, set `EPOCHS = 50` and re-commit — a clean 50-epoch run from scratch. Equal or slightly better quality, but redoes the full ~6 h.

## 1. GPU + CPU

In [ ]:
import os, torch
print('CPUs:', os.cpu_count(), '| CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' ', torch.cuda.get_device_name(i))

## 2. Install (no -U, to keep Kaggle's torch / GPU compatibility)

In [ ]:
!pip install -q ultralytics roboflow
import torch, ultralytics, roboflow
print('torch', torch.__version__, '| ultralytics', ultralytics.__version__)

## 3. Roboflow API key

In [ ]:
import os
ROBOFLOW_API_KEY = ''
try:
    from kaggle_secrets import UserSecretsClient
    ROBOFLOW_API_KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
except Exception:
    pass
if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')
assert ROBOFLOW_API_KEY, (
    'No Roboflow API key found. Add a Kaggle Secret named ROBOFLOW_API_KEY '
    '(Add-ons -> Secrets, then attach it to this notebook), or set the '
    'ROBOFLOW_API_KEY environment variable. Get your key at '
    'https://app.roboflow.com (Settings -> API).'
)

## 4. Download the dataset

In [ ]:
import shutil
shutil.rmtree('/kaggle/working/ppe', ignore_errors=True)
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('segp-fcn6m').project('ppe-yezzu-fwbjo')
dataset = project.version(1).download('yolov8', location='/kaggle/working/ppe')
print('Downloaded to:', dataset.location, '|', os.listdir(dataset.location))

## 5. Fix data.yaml

In [ ]:
import glob, yaml
yml = glob.glob('/kaggle/working/**/data.yaml', recursive=True)[0]
DATA_DIR = os.path.dirname(yml)
spec = yaml.safe_load(open(yml))
spec['path'] = DATA_DIR
spec['train'], spec['val'], spec['test'] = 'train/images', 'valid/images', 'test/images'
yaml.safe_dump(spec, open(yml, 'w'), sort_keys=False)
print('data.yaml:', yml, '| classes:', spec['names'])
for k in ('train', 'val', 'test'):
    print(f'  {k}: {os.path.isdir(os.path.join(DATA_DIR, spec[k]))}')

## 6. Find your uploaded best.pt

In [ ]:
cands = glob.glob('/kaggle/input/**/best.pt', recursive=True) + glob.glob('/kaggle/input/**/*.pt', recursive=True)
assert cands, 'No .pt under /kaggle/input — upload best.pt as a Dataset and Add Input (see top markdown).'
BEST_IN = cands[0]
print('Starting weights:', BEST_IN)

## 7. Continue training (+20 epochs, low LR)

Loads your weights and refines them. `optimizer='SGD'` + low `lr0` is used so the small learning rate is actually honoured (the default `optimizer='auto'` would ignore `lr0` and pick its own, higher rate — wrong for a continue).

In [ ]:
from ultralytics import YOLO

EPOCHS = 20
IMGSZ = 640
BATCH = 48          # 96 also fine on T4; data-loading is the limit, not memory
DEVICE = 0          # '0,1' to use both T4s
PROJECT_DIR = '/kaggle/working/runs'
RUN_NAME = 'ppe_s_cont'

model = YOLO(BEST_IN)
model.train(
    data=yml,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    optimizer='SGD',
    lr0=0.001,        # gentle: ~1/10 of the from-scratch rate
    lrf=0.01,
    warmup_epochs=1.0,
    patience=10,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)
BEST = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'best.pt')
print('Refined checkpoint:', BEST)

## 8. Evaluate on the test split (compare to your 30-epoch numbers)

In [ ]:
def evaluate(weights, tta=False):
    m = YOLO(weights)
    res = m.val(data=yml, split='test', imgsz=IMGSZ, augment=tta, device=0, verbose=False)
    box = res.box
    names = res.names
    idx = list(box.ap_class_index)
    p, r, f1 = list(box.p), list(box.r), list(box.f1)
    ap50, ap = list(box.ap50), list(box.ap)
    print('=' * 78)
    print(f" Per-class metrics on TEST{'  (TTA)' if tta else ''}")
    print('=' * 78)
    print(f"{'class':<12}{'precision':>11}{'recall':>10}{'F1':>9}{'mAP@50':>10}{'mAP50-95':>11}{'  90%?':>8}")
    n90 = 0
    for j, ci in enumerate(idx):
        name = names[int(ci)]
        row = dict(precision=float(p[j]), recall=float(r[j]), f1=float(f1[j]),
                   map50=float(ap50[j]), map5095=float(ap[j]))
        hit = all(row[k] >= 0.90 for k in ('precision', 'recall', 'f1', 'map50'))
        n90 += hit
        f = lambda x: f'{x*100:6.1f}%'
        print(f"{name:<12}{f(row['precision']):>11}{f(row['recall']):>10}{f(row['f1']):>9}"
              f"{f(row['map50']):>10}{f(row['map5095']):>11}{('  YES' if hit else ''):>8}")
    mp, mr = float(box.mp), float(box.mr)
    mf1 = 2*mp*mr/(mp+mr) if (mp+mr) else 0.0
    print('-' * 78)
    print(f"{'ALL (mean)':<12}{mp*100:10.1f}%{mr*100:9.1f}%{mf1*100:8.1f}%"
          f"{float(box.map50)*100:9.1f}%{float(box.map)*100:10.1f}%")
    print(f"\nClasses clearing 90% on P/R/F1/mAP@50: {n90}/{len(idx)}")
    return res

print('30-epoch baseline was: mAP@50 97.9%, mAP@50-95 84.2%, all 5 classes >=90%.\n')
_ = evaluate(BEST)

## 9. Save the refined model for download

In [ ]:
shutil.copy(BEST, '/kaggle/working/best_refined.pt')
print('Saved /kaggle/working/best_refined.pt — download from the Output tab.')
# Keep whichever scores higher on mAP@50-95 (this refined one vs your 30-epoch best.pt).